# Profils des enquêtés

## Les pratiques de travail

In [13]:
import pandas as pd

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket
import scipy.stats

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [14]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [15]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")

list_affil = pd.read_csv("../list_affiliation.csv", sep =",")
df0 = df0.loc[~df0.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2"])].merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")

PermissionError: Forbidden

In [4]:

df_col = pd.read_csv("../../le_questionnaire/dico_variable.csv", sep = ",")


In [5]:
list_nominal_simple = [x for x in df_col.label.loc[(df_col.type.isin(["simple_nominal", "booléen","ordinal"]))]]


In [6]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [7]:
def grouped_question(data, column, index = "q45_clé"):
    """


    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [8]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[y], margins = False, normalize=False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index(), cross_tab0

In [9]:
df0[["q45_clé","q26_env_travail"]]

NameError: name 'df0' is not defined

In [10]:
df_exp, gb_data = split_multiple_choices(df0[["q45_clé","q26_env_travail"]], "q26_env_travail", "q45_clé", sep = '|')
print(gb_data[["q26_env_travail","nb","freq","total"]].to_markdown(index=False))

NameError: name 'df0' is not defined

La première question interrogeant les pratiques de travail des enquêtés porte sur le lieu habituel de travail en classant par ordre de priorité les options suivantes :
- "À mon domicile"
- "Dans mon bureau sur site"
- "Autre"

90% des personnes interrogés disent travailler le plus souvent à leur domicile ({numref}`env_travail`). C'est aussi le lieu le plus souvent cité en premier. 71% des répondants travaillent ainsi prioritairement à leur domicile. Tandis que le bureau est en moyenne classé deuxième dans l'ordre des priorités.  

Parmis les autres lieux cités, on trouve tout d'abord les bibliothèques et les autres sites (CNRS, Campus Condorcet, partenaires de recherche). On note également que plusieurs répondants ont indiqué qu'ils travaillaient un peu partout, là où ils pouvaient faute de bureau disponible sur le campus. Ces réponses recoupent les résultats observés concernant les services à développer où la construction d'espace de travail avaient été choisi par plus de 80% des répondants ({numref}`11_service_to_develop`).


```{table} Où travaillez-vous le plus souvent ?
:name: env_travail


|   rang |   A mon domicile         |   Dans mon bureau sur site         |   Autre, précisez         |
|-------:|-------------------------:|-----------------------------------:|--------------------------:|
|      1 |               85 (71,4%) |                         29 (24,4%) |                 5  (4,2%) |
|      2 |               20 (16,8%) |                         36 (20,3%) |                12 (10,1%) |
|      3 |                3  (2,5%) |                          5  (4,2%) |                14 (11,8%) |
|Total   |              108 (90,8%) |                         70 (58,8%) |                31 (26,1%) |

```

In [11]:
#On compte le nombre de choix par personne

nb_choix = df_exp.groupby(["q45_clé"]).agg(nb=("q26_env_travail", "size")).reset_index()

np.mean(nb_choix.nb)
nb_choix.groupby(["nb"]).agg(freq=("q45_clé", "size")).reset_index()

NameError: name 'df_exp' is not defined

In [12]:
### Traitement des questions ordonnées : on calcule la fréquence par rang pour chaque option, puis le rang moyen
list_rank= []

for cle in df_exp.q45_clé.unique():
    dtmp = df_exp.loc[df_exp.q45_clé==cle]
    for n, r in enumerate(dtmp.q26_env_travail):
        dict_rank={"q45_clé":cle,
                   "q26_env_travail":r,
                   "rang":n+1}
        list_rank.append(dict_rank)
list_rank
    
df_rank = pd.DataFrame.from_dict(list_rank)  

df_rank

NameError: name 'df_exp' is not defined

In [82]:
d_class = df_rank.groupby(["rang","q26_env_travail"]).agg(nb=("q45_clé","size")).reset_index()
d_class["freq"] = round(d_class.nb/119*100, 1)
d_class_tab = pd.pivot(d_class, index= "rang", columns="q26_env_travail", values="freq").reset_index()
print(d_class_tab[["rang","A mon domicile",  "Dans mon bureau sur site",  "Autre, précisez"]].to_markdown(index=False))
#df_rank.groupby(["q26_env_travail","rang"]).agg(nb=("q45_clé","size"))

|   rang |   A mon domicile |   Dans mon bureau sur site |   Autre, précisez |
|-------:|-----------------:|---------------------------:|------------------:|
|      1 |             71.4 |                       24.4 |               4.2 |
|      2 |             16.8 |                       30.3 |              10.1 |
|      3 |              2.5 |                        4.2 |              11.8 |


In [84]:
for x in df0.q26_1_autre_env.loc[~df0.q26_1_autre_env.isna()]:
    print(x)

Bibliothèque 
Dans les locaux de mes partenaires de recherche
Bibliothèques 
Dans le métro
bibliotheque
Réunions avec les partenaires hospitaliers sur leur site
SITE CNRS
Bibliothèque, archives
terrain 
bureau de collègues dans d'autres universités
Bibliothèque
bibliothèque et archives
bureau hors P8
bibliothèques
Bureau MSH OU BIBLIOTHÈQUE MAIS JE MANQUE CRUELLEMENT D'UN ESPACE DE TRAVAIL 
Où je peux, puisque nous n'avon pas de bureaux. Lorsque je peux, j'évite le campus car : difficultés d'accès, de connexion ; risque que les salles soient déjà prises, etc. Cela est très bloquant pour tout travail d'équipe.
partout
Au laboratoire CEMTI
où je peux, je n'ai pas de bureau à la fac (15 m2 pour 6 titulaires)
colloques, train, hotel
dans des bibliothèque universitaires, au laboratoire  
Transports en commun, salles d'attente...
Lieux de recherche 
Campus Condorcet (chercheur associé à l'Ined)
Bibliothèques P8 et ailleurs
Dans un bureau sur site partagé avec 5 autres collègues, trop petit, 

In [87]:
dict_autre_env ={
    "Bibliothèque" :"Bibliothèque",
"Dans les locaux de mes partenaires de recherche":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèques":"Bibliothèque",
"Dans le métro":"Métro",
"bibliotheque":"Bibliothèque",
"Réunions avec les partenaires hospitaliers sur leur site":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"SITE CNRS":"Autres sites (CNRS, partenaires)",
"Bibliothèque, archives":"Bibliothèque",
"terrain" :"Terrains",
"bureau de collègues dans d'autres universités":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèque":"Bibliothèque",
"bibliothèque et archives":"Bibliothèque",
"bureau hors P8":"Autres sites (CNRS, partenaires)",
"bibliothèques":"Bibliothèque",
"Bureau MSH OU BIBLIOTHÈQUE MAIS JE MANQUE CRUELLEMENT D'UN ESPACE DE TRAVAIL":"Autres sites (Campus Condorcet, CNRS, partenaires)|Bibliothèque",
"Où je peux, puisque nous n'avon pas de bureaux. Lorsque je peux, j'évite le campus car : difficultés d'accès, de connexion ; risque que les salles soient déjà prises, etc. Cela est très bloquant pour tout travail d'équipe.":"Où je peux",
"partout":"Où je peux",
"Au laboratoire CEMTI":"Laboratoire",
"où je peux, je n'ai pas de bureau à la fac (15 m2 pour 6 titulaires)":"Où je peux",
"colloques, train, hotel":"Déplacement",
"dans des bibliothèque universitaires, au laboratoire":"Bibliothèque|Laboratoire",
"Transports en commun, salles d'attente...":"Où je peux",
"Lieux de recherche" :"Où je peux",
"Campus Condorcet (chercheur associé à l'Ined)":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèques P8 et ailleurs":"Bibliothèque|Où je peux",
"Dans un bureau sur site partagé avec 5 autres collègues, trop petit, peu propice à la concentration et pas adapté aux réunions en visio":"Laboratoire",
"Bibliothèques.. Je trouve incongru la question du bureau dont nous ne disposons pas du tout (un peu plus depuis la construction de la Maison de la Recherche mais la chose n'a visiblement pas été pensée jusqu'au bout ou vraiment pour les enseignants chercheurs":"Bibliothèque"
}

In [89]:
df0["q26_1_autre_rec"] = df0.q26_1_autre_env.map(dict_autre_env.get)


,q26_1_autre_rec
0,None
1,None
2,None
3,None
4,None
...,...
114,None
115,Bibliothèque|Où je peux
116,None
117,None


In [97]:
df0

,Unnamed: 0,qno_obs,q1_so_principles,q2_hal_depot,q3_octavi_depot,q4_diff_data,q4_autres_entrepots,q5_r,q5_python,q5_excel,...,q16_ocr_help,q16_network_analysis_help,q16_odette_atom_help,q16_caqdas_help,q16_carto_help,q16_sig_help,q16_sgbd_help,q16_autre_help,q16_1_other_software_help,q26_1_autre_rec
0,0,1,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,...,"Oui, débutant",Non,Non,Non,Non,Non,"Oui, avancé",Non,NaN,None
1,1,2,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,...,"Oui, débutant",Non,Non,Non,Non,Non,Non,Non,NaN,None
2,2,3,"Oui, tout à fait","Oui, plusieurs fois",Je ne connais pas Octaviana,Non,NaN,Non,Non,Oui,...,Non,Non,Non,"Oui, débutant",Non,Non,Non,"Oui, débutant",Logiciel de traitement d'entretiens et de vidéos,None
3,3,4,"Oui, un peu","Oui, une fois",Non,"Autre, précisez",OSF,Oui,Non,Non,...,Non,Non,Non,Non,Non,Non,Non,Non,NaN,None
4,4,6,"Oui, un peu","Oui, systématiquement",Non;Je ne connais pas Octaviana,Non,NaN,Oui,Non,Oui,...,Non,Non,Non,Non,Non,Non,Non,Non,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,114,117,"Oui, un peu","Oui, plusieurs fois",Non,"Autre, précisez",OSF,Oui,Oui,Oui,...,Non,Non,"Oui, débutant","Oui, débutant","Oui, débutant",Non,"Oui, débutant",Non,NaN,None
115,115,118,"Oui, un peu","Oui, systématiquement",Non,Non,NaN,Oui,Non,Oui,...,"Oui, débutant","Oui, débutant","Oui, débutant","Oui, débutant","Oui, débutant","Oui, débutant","Oui, débutant",Non,NaN,Bibliothèque|Où je peux
116,116,119,"Oui, un peu","Oui, plusieurs fois",Non,Non,NaN,Oui,Non,Oui,...,Non,"Oui, débutant",Non,Non,Non,"Oui, avancé","Oui, débutant",Non,NaN,None
117,117,120,Non,"Oui, plusieurs fois",Non;Je ne connais pas Octaviana,"Autre, précisez",Mendeley data,Non,Oui,Oui,...,"Oui, débutant","Oui, débutant","Oui, débutant","Oui, débutant","Oui, débutant","Oui, débutant","Oui, avancé","Oui, débutant",​-,None


In [98]:
# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

KeyboardInterrupt: 

In [92]:
df_col.loc[df_col.label.str.contains("q26_1")].to_dict()

{'name': {100: '113. Autre, précisez'},
 'label': {100: 'q26_1_autre_env'},
 'group': {100: '13_env_travail'},
 'personal_data': {100: False},
 'type': {100: 'texte_libre'},
 'opened_question': {100: True},
 'type_panda': {100: 'object'},
 'no_question': {100: 33.0},
 'question': {100: '1/ Où travaillez-vous le plus souvent ?'},
 'question_family': {100: 'q26'},
 'comment': {100: nan}}

In [95]:
dic_new_var = {'name':'113. Autre, précisez',
               'label':'q26_1_autre_rec',
               'group':'13_env_travail',
               'personal_data':False,
               'type':'nominal_multiple',
               'opened_question':False,
               'type_panda':df0.q26_1_autre_rec.dtypes,
               'no_question':113,
               'question':"Où travaillez-vous le plus souvent? Si autre, précisez",
               'question_family':'q26',
               'comment':"recode les autre lieux"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col,new_variable], ignore_index = True)
df_col1

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
162,62. Logiciels de SIG (ex. QGis),q16_sig_help,9_help_outil,False,ordinal,False,object,19.0,7/ Ressentez-vous le besoin d'un accompagnemen...,q16,NaN
163,63. Système de gestion de base de données,q16_sgbd_help,9_help_outil,False,ordinal,False,object,19.0,7/ Ressentez-vous le besoin d'un accompagnemen...,q16,NaN
164,64. Autre,q16_autre_help,9_help_outil,False,ordinal,False,object,19.0,7/ Ressentez-vous le besoin d'un accompagnemen...,q16,NaN
165,65. other_accomp_software,q16_1_other_software_help,9_help_outil,False,texte_libre,True,object,20.0,"7.1/ Si Autre, précisez lequel (lesquels) ?",q16,NaN
